# Lecture 3: Quantum Denoising & Advanced Features
## Application to Medical Image Processing

In this final lecture, we will see how MHRQI can be used for a real-world task: **Denoising OCT (Optical Coherence Tomography) scans**.

---

### 1. The Denoising Problem in Medical Imaging

Medical images like OCT scans are often corrupted by **speckle noise**. MHRQI's hierarchical structure allows us to apply a unique quantum denoising technique based on **hierarchical consistency**.

### 2. Setting Up a Noisy Scenario

Let's load a real OCT scan from the resources and prepare it for MHRQI.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from mhrqi import MHRQI
from mhrqi.utils import general as utils

# Load sample OCT scan
img_raw = cv2.imread("../resources/drusen1.jpeg", cv2.IMREAD_GRAYSCALE)
img_resized = cv2.resize(img_raw, (16, 16)) / 255.0

plt.imshow(img_resized, cmap="gray")
plt.title("Original OCT Scan (16x16)")
plt.show()

### 3. Enabling MHRQI Denoising

To enable denoising, we simply call `apply_denoising()` on our model before simulation. This adds an **ancilla-based consistency check** to the quantum circuit.

In [ ]:
size = 16
depth = int(np.log2(size))
h_matrix = utils.generate_hierarchical_coord_matrix(size, d=2)

# Initialize model
model = MHRQI(depth=depth)
model.lazy_upload(h_matrix, img_resized)

# APPLY DENOISING
model.apply_denoising()

print("Denoising extension enabled.")

### 4. Sibling Smoothing

When we reconstruct the image from a denoised MHRQI model, the `MHRQIResult` object uses an algorithm called **Sibling Smoothing**. 

It uses the "confidence" of the quantum consistency check to decide whether to trust a pixel's value or blend it with its hierarchical neighbors.

In [ ]:
# Simulate
result = model.simulate(shots=10000)

# Reconstruction automatically applies sibling smoothing if denoising was enabled
denoised_recon = result.reconstruct(use_denoising_bias=True)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_resized, cmap="gray")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(denoised_recon, cmap="gray")
plt.title("MHRQI Denoised")
plt.show()

### 5. Conclusion

MHRQI provides a powerful, modular framework for quantum image processing. By combining hierarchical representation with consistency-based post-selection, it achieves superior results in noisy environments.

**Key Takeaways**:
- MHRQI uses **Quad-trees** to represent image hierarchy.
- The Core API provides an **object-oriented** way to build and simulate these circuits.
- **Denoising** is a built-in feature that leverages hierarchical spatial correlations.

---
**Thank you for participating in the MHRQI Lecture Series!**